# DWR Outlier Detection Tests + Machine-Learning Ready Dataset

Five groups across multiple state and federal organizations monitor water quality and quantity through a network of instruments located within water bodies across California. [Here](https://cdec.water.ca.gov/webgis/?appid=cdecstation) is a map of all the instruments (select sensor type: other).

Currently, all five organizations implement different pipelines to read, process, store, and serve the data. We propose to create a prototype machine-learning ready dataset that demonstrates a unified pipeline to create a consistent, reproducible, and easily accessible datataset. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime as dt_obj
from IPython.display import display
from scipy import stats
from pandas.api.types import is_datetime64_any_dtype
from tsfresh import feature_extraction
from tsfresh.utilities.dataframe_functions import roll_time_series
 
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 3000)

## Data processing

To process the data, we will implement standardized outlier detection methods as outlined in the [Outlier Detection Test Best Practices](https://docs.google.com/document/d/1YmXWDMHCKnWniQ4sMkkT5g5j5H5TmT3f/edit?usp=drive_link&ouid=112329730124000397523&rtpof=true&sd=true) guide. Group 3 includes suggested tests, which we can supplement with other ideas (e.g. [Talagala+2019](https://doi.org/10.1029/2019WR024906)). 

### Ingest sample data

In [2]:
def get_data_from_local_directory(filename, path):
    """
        Get data from a local directory.

        Parameters
        ----------
        filename : str
            Name of file to get from local directory.
        path : str
            Path to local directory.
        
        Returns
        -------
        df : pandas.DataFrame
            Dataframe containing data from local directory.
        
        Examples
        --------
        >>> get_data_from_local_directory('test.csv', 'data/')
    """
    file_with_path = path + filename
    df = pd.read_csv(file_with_path)
    
    return df

In [3]:
df = get_data_from_local_directory(filename='sample_data_BKS.csv', path='sample_data/')

In [4]:
df

,STATION_ID,DURATION,SENSOR_NUMBER,SENSOR_TYPE,DATE TIME,OBS DATE,VALUE,DATA_FLAG,UNITS
0,BKS,D,62,PH VAL,19890203 0000,19890203 0100,---,,PH
1,BKS,D,62,PH VAL,19890204 0000,19890204 0000,---,,PH
2,BKS,D,62,PH VAL,19890205 0000,19890205 0000,---,,PH
3,BKS,D,62,PH VAL,19890206 0000,19890206 0000,---,,PH
4,BKS,D,62,PH VAL,19890207 0000,19890207 0000,---,,PH
...,...,...,...,...,...,...,...,...,...
12914,BKS,D,62,PH VAL,20241207 0000,20241207 0000,7.4,,PH
12915,BKS,D,62,PH VAL,20241208 0000,20241208 0000,7.5,,PH
12916,BKS,D,62,PH VAL,20241209 0000,20241209 0000,7.5,,PH
12917,BKS,D,62,PH VAL,20241210 0000,20241210 0000,7.5,,PH


Convert all string values of '---' to `np.Nan`

In [5]:
df.replace(to_replace='---', value=np.nan, inplace=True)

Convert string values in `DATE TIME` column to datetime object, rename it `TIME`, and set it as the index.

In [6]:
df['TIME']=pd.to_datetime(df['DATE TIME'], format='%Y%m%d %H%M').reset_index(drop=True)

In [7]:
df = df.set_index('TIME')

Convert all string values of numbers to ints for the column `VALUE`

In [8]:
df.VALUE = pd.to_numeric(df.VALUE)

## Group 3 Tests

The following test uses a kNN method described in [Talagala+2019](http://dx.doi.org/10.1029/2019WR024906). Group 3 of the [Outlier Detection Test Best Practices](https://docs.google.com/document/d/1YmXWDMHCKnWniQ4sMkkT5g5j5H5TmT3f/edit) document suggests using machine-learning methods for outlier detection tests.

#### Step 1. Read the labelled data.

* Right now, we are reading data that was artificially flagged using the outlier detection tool. Ideally, we would have a list of real outliers.
* Right now, we are using univariate data (pH). Ideally, we could detect outliers using multivariate methods.

In [9]:
df = get_data_from_local_directory(filename='flagged_sample_data_BKS.csv', path='sample_data/')

In [10]:
# Convert to datetime object
df['DATE TIME']=pd.to_datetime(df['DATE TIME'], format='%Y-%m-%d').reset_index(drop=True)

In [11]:
# Create a single binary flag
df['Flag'] = np.where(((df['VALUE_failed_modified_z_score_test'] == 'failed') | 
                       (df['VALUE_failed_spike_detection_test'] == 'failed') |
                       (df['VALUE_failed_value_gap_test'] == 'failed') |
                       (df['VALUE_failed_z_score_test'] == 'failed') |
                       (df['VALUE_failed_flat_line_test'] == 'failed') |
                       (df['DATE TIME_failed_time_gap_test'] == 'failed') |
                       (df['VALUE_failed_tukey_iqr_test'])), 1, 0)

In [47]:
# if VALUE has a NaN then drop it
df_clean = df_subset[df_subset['VALUE'].notna()].reset_index(drop=True)

#### Step 2. Derive features from the data.

* Right now, we are creating simple features in the temporal domain. In the future, we can look at the spectral domain (e.g. Fourier space). Future ideas also include computing features with convolutional kernels (e.g. [ROCKET](https://www.aeon-toolkit.org/en/stable/examples/transformations/rocket.html)).

In [46]:
df_subset = df[['VALUE', 'DATE TIME', 'Flag']]

In [48]:
# Create a one-week rolling time window
one_week = pd.Timedelta('7 days')
df_rolled = df_clean[['VALUE', 'DATE TIME']].rolling(window = one_week, on='DATE TIME')

In [49]:
# Calculate absolute maximum
def calculate_absolute_maximum(n):
    return feature_extraction.feature_calculators.absolute_maximum(n)

# Calculate the absolute energy
def calculate_absolute_energy(n):
    return feature_extraction.feature_calculators.abs_energy(n)

# Calculate a measure of complexity
def calculate_absolute_complexity(n):
    return feature_extraction.feature_calculators.cid_ce(n, normalize = False)

In [52]:
df_absolute_energy = df_rolled.apply(calculate_absolute_energy)
df_absolute_energy = df_absolute_energy.rename(columns={'VALUE':'Absolute energy'})

In [53]:
df_absolute_maximum = df_rolled.apply(calculate_absolute_maximum)
df_absolute_maximum = df_absolute_maximum.rename(columns={'VALUE':'Absolute maximum'})

df_absolute_complexity = df_rolled.apply(calculate_absolute_complexity)
df_absolute_complexity = df_absolute_complexity.rename(columns={'VALUE':'Absolute complexity'})

In [73]:
df = pd.concat([df_absolute_energy, df_absolute_maximum, df_absolute_complexity], join='outer').drop_duplicates().reset_index(drop=True)

In [78]:
df_clean['Absolute complexity'] = df_absolute_complexity['Absolute complexity']
df_clean['Absolute maximum'] = df_absolute_maximum['Absolute maximum']
df_clean['Absolute energy'] = df_absolute_energy['Absolute energy']

In [79]:
df_clean

,VALUE,DATE TIME,Flag,Absolute complexity,Absolute maximum,Absolute energy
0,7.6,1992-10-30,1,0.000000,7.6,57.76
1,7.8,1992-10-31,1,0.200000,7.8,118.60
2,7.8,1992-11-01,1,0.200000,7.8,179.44
3,7.6,1992-11-02,1,0.282843,7.8,237.20
4,7.8,1992-11-03,1,0.346410,7.8,298.04
...,...,...,...,...,...,...
9373,7.4,2024-12-06,1,0.100000,7.5,392.26
9374,7.4,2024-12-07,1,0.100000,7.5,390.77
9375,7.5,2024-12-08,1,0.141421,7.5,390.77
9376,7.5,2024-12-09,1,0.141421,7.5,390.77


#### Step 3. Split the data into training and testing data.

In [80]:
df_train = df_clean.iloc[0:int(len(df)*0.7)]
df_test = df_clean.iloc[int(len(df)*0.7):]

## Group 1 Tests

The following six tests comprise Group 1 of the [Outlier Detection Test Best Practices](https://docs.google.com/document/d/1YmXWDMHCKnWniQ4sMkkT5g5j5H5TmT3f/edit) document. 

### Time Gap test

The Time Gap test to see if there is a gap in the data based on an expected time frequency (e.g., 15 minutes, 1 hour, etc.).

The input variable `cadence` defines the nominal cadence of the time series. This test is less useful than preferred, because the sample data contains an entry for every time stamp at appropriate cadence, but includes `np.nan` for missing data. A more useful test is a value gap test (below).

In [ ]:
def time_gap_test(ts: pd.Series, number: int, unit: str) -> pd.DataFrame:
    """
    Apply a time gap test.
    
    Parameters
    ----------
    ts : pd.Series
       A pandas series with a datetime index.   
    
    number : int
        Number (ex: 1, 2) to define the expected cadence of the data.

    unit : str
        String (ex: days, hours) to define the time unit to measure. pandas.Timedelta must support this.

    Returns
    -------
    pd.Series
        A series where True designates missing values and False designates otherwise.

    Examples
    --------
    >>> df_time_gap = time_gap_test(df, cadence=pd.Timedelta('1 days'))

    """
    if df.empty:
        raise ValueError('No input data.')
    if not is_datetime64_any_dtype(ts):
        raise ValueError('Input date column does not have a valid datetime data type.')
    try:
        item_try = pd.Timedelta(number, unit)
    except ValueError as item_try:
        print(str(item_try))
        print("Use 'hours', 'minutes', or 'days'.")
        return np.nan 

    cadence = pd.Timedelta(number, unit)
    output_ts = ts.diff() != cadence
    
    return output_ts

In [ ]:
output_ts = time_gap_test(ts=df.index, number=1, unit='days')

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(output_ts, edgecolor=None)

In [ ]:
def value_gap_test(ts: pd.Series) -> pd.Series:
    """
    Apply a value gap test to a time series.
    
    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.

    Returns
    -------
    pd.Series
        A series where True designates missing values and False designates otherwise.

    Examples
    --------
    >>> output_ts = value_gap_test(df.VALUE)

    """
    if ts.empty:
        raise ValueError('No input data.')
    output_ts = ts.isna()
    return output_ts

In [ ]:
output_ts = value_gap_test(ts=df.VALUE)

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(output_ts, edgecolor=None)

### Basic and Tuned Gross range test

Basic Gross Range Test: Checks if data point exceeds sensor minimum or maximum values.*  
Tuned Gross Range Test: A variation on the Gross Range Test, where the thresholds are determined and adjusted by the operator on a monthly or seasonal basis, or some other predefined time period.

The input variable `bound` can define sensor maximum and minimum values or operator-determined thresholds. Therefore the gross range test below satisfies both tests (4) and (5) in Group 1 of the [Outlier Detection Test Best Practices](https://docs.google.com/document/d/1YmXWDMHCKnWniQ4sMkkT5g5j5H5TmT3f/edit) document.

In [ ]:
def gross_range_test(ts: pd.Series, minimum: float, maximum: float) -> pd.DataFrame:
    """
    Apply a gross range test to a time series.

    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.

    minimum : float
        The lower bounds for the test (can be None if maximum is not None).

    maximum : float
        The upper bounds for the test (can be None if minimum is not None).

    Returns
    -------
    pandas.Series
        A time series where True indicates values outside the bounds and False indicates otherwise.
    
    Examples
    --------
    >>> output_ts = gross_range_test(ts=df_in, minimum=0, maximum=100))

    """
    if ts.empty:
        raise ValueError('No input data.')
    if minimum is None:
        raise ValueError('Minimum must be specified.')
    if maximum is None:
        raise ValueError('Maximum must be specified.')
    if minimum > maximum:
        raise ValueError('Maximum value cannot be less than minimum value.')

    output_ts = (ts > maximum) | (ts <= minimum)

    return output_ts

In [ ]:
output_ts = gross_range_test(ts=df.VALUE, minimum=4, maximum=8)

In [ ]:
df_gross_range = pd.DataFrame(output_ts)
df_gross_range['pH'] = df.VALUE

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(df_gross_range, x='TIME', y='pH', hue='VALUE', edgecolor=None)

### Flat line test

The flat line test compares the present observation to several previous observations to identify a continuously repeated observation of the same value, which is a common result when sensors and/or data collection platforms fail.

In [ ]:
def flat_line_test(ts: pd.Series, number_of_repeated_values: int = 3) -> pd.DataFrame:
    """
    Apply a flat line test to a time series.

    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.
    
    number_of_repeated_values : int
        The number of consecutive repeated values to indicate an instrumental anomaly.

    Returns
    -------
    pandas.DataFrame
        Outliers flagged as True or False.
    
    Examples
    --------
    >>> df_out = flat_line_test(df_in)

    """
    if ts.empty:
        raise ValueError('No input data.')
    mask = ts.ne(ts.shift())
    counts = ts.groupby(mask.cumsum()).transform('count')
    output_ts = counts >= number_of_repeated_values
    return output_ts

In [ ]:
output_ts = flat_line_test(ts=df.VALUE, number_of_repeated_values=5)

In [ ]:
df_flat_line = pd.DataFrame(output_ts)
df_flat_line['pH'] = df.VALUE

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(df_flat_line, x='TIME', y='pH', hue='VALUE', edgecolor=None)

## Group 2 Tests

The following six tests comprise Group 2 of the [Outlier Detection Test Best Practices](https://docs.google.com/document/d/1YmXWDMHCKnWniQ4sMkkT5g5j5H5TmT3f/edit) document. 

### Z-score and Modified Z-score tests

The Z-score calculates the number of standard deviations from the mean per point in the time series. Taken from Scipy; see [stats.zscore()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.zscore.html).

In [ ]:
def z_score_test(ts: pd.Series, number_of_standard_deviations: int=3) -> pd.Series:
    """
    Apply the Scipy Z-Score test to a time series. Flag values based on the number of standard deviations from the mean.

    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.
    
    number_of_standard_deviations : int, optional
        The number of standard deviations from the mean to flag. Default is 3.

    Returns
    -------
    pandas.Series
        Outliers flagged as True or False.
    
    Examples
    --------
    >>> output_ts = z_score_test(ts=df.VALUE)

    """
    if ts.empty:
        raise ValueError('No input data.')
    z_score = stats.zscore(ts, nan_policy='omit')
    output_ts = np.abs(z_score) >= number_of_standard_deviations
    return output_ts

In [ ]:
output_ts = z_score_test(ts=df.VALUE)

In [ ]:
df_z_score = pd.DataFrame({"TIME":df.index, "VALUE":output_ts, "pH":df.VALUE})

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(df_z_score, x='TIME', y='pH', hue='VALUE', edgecolor=None).set_title('Outlier Detection Method: \nZ-Score')
ax.set(ylabel='pH')

In [ ]:
def modified_z_score_test(ts: pd.Series, median_absolute_deviation: float=4) -> pd.Series:
    """
    Apply the Scipy Z-Score test to a time series. Flag values based on the number of standard deviations from the mean.

    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.
    
    median_absolute_deviation : float, optional
        The threshold defining the maximum allowable median absolute deviation. Default is 4.

    Returns
    -------
    pandas.Series
        Outliers flagged as True or False.
    
    Examples
    --------
    >>> output_ts = z_score_test(ts=df.VALUE)

    """
    if ts.empty:
        raise ValueError('No input data.')
    modified_z_score = (stats.norm.ppf(3/4) * (ts - ts.median())) / (stats.median_abs_deviation(ts, nan_policy='omit'))
    output_ts = np.abs(modified_z_score) >= median_absolute_deviation
    return output_ts

In [ ]:
output_ts = modified_z_score_test(ts=df.VALUE)

In [ ]:
df_modified_z_score = pd.DataFrame({"TIME":df.index, "VALUE":output_ts, "pH":df.VALUE})

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(df_z_score, x='TIME', y='pH', hue='VALUE', edgecolor=None).set_title('Outlier Detection Method: \n Modified Z-Score')
ax.set(ylabel='pH')

### Tukey's Interquartile Range test

Tukey's Interquartile Range test calculates values that fall outside of 1.5 times the interquartile range between the first and third quartiles.

In [ ]:
def tukey_iqr_test(ts: pd.Series) -> pd.DataFrame:
    """
    Apply Tukey's IQR test to a time series.

    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.

    Returns
    -------
    pandas.DataFrame
        Outliers flagged as True or False.
    
    Examples
    --------
    >>> df_out = tukey_iqr_test(ts)

    """
    if ts.empty:
        raise ValueError('No input data.')
    c = stats.norm.ppf(3 / 4) - stats.norm.ppf(1 / 4)
    quantiles = ts.quantile([0.25, 0.75])
    np.squeeze(np.diff(quantiles, axis=0) / c)
    iqr = quantiles[0.75] - quantiles[0.25]
    upper_limit = quantiles[0.75] + (iqr*1.5)
    lower_limit = quantiles[0.25] - (iqr*1.5)
    output_ts = ((ts > upper_limit) | (ts < lower_limit))
    return output_ts

In [ ]:
output_ts = tukey_iqr_test(ts=df.VALUE)

In [ ]:
df_tukey_iqr_score = pd.DataFrame({"TIME":df.index, "VALUE":output_ts, "pH":df.VALUE})

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(df_tukey_iqr_score, x='TIME', y='pH', hue='VALUE', edgecolor=None).set_title('Outlier Detection Method: \nTukey\'s Interquartile Range')
ax.set(ylabel='pH')
#fig.savefig('tukeyiqr.png', dpi=300, transparent=True, bbox_inches='tight')

### Spike Detection Test

The Spike Detection Test identifies a spike, defined as a value greater than the mean*factor of the adjacent values, in a time series.

In [ ]:
def spike_detection_test(ts: pd.Series, factor: float = 1.05) -> pd.DataFrame:
    """
    Identify a spike, defined as a value greater than the mean*factor of the adjacent values, in a time series.

    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.

    factor : float, optional
        The factor by which the mean of adjacent values is multiplied to determine a spike.

    Returns
    -------
    pandas.DataFrame
        Outliers flagged as True or False.
    
    Examples
    --------
    >>> df_out = spike_detection_test(ts)

    """
    if ts.empty:
        raise ValueError('No input data.')
    mean_adjacent = (ts.shift(1) + ts.shift(-1)) / 2
    output_ts = ts > factor*mean_adjacent
    return output_ts

In [ ]:
output_ts = spike_detection_test(ts=df.VALUE)

In [ ]:
df_spike_detection = pd.DataFrame({"TIME":df.index, "VALUE":output_ts, "pH":df.VALUE})

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(df_spike_detection, x='TIME', y='pH', hue='VALUE', edgecolor=None).set_title('Outlier Detection Method: \nSpike Detection')
ax.set(ylabel='pH')
#fig.savefig('spike.png', dpi=300, transparent=True, bbox_inches='tight')

### Rate of Change Test

The Rate of Change Test compares the mean value of the previous N points to the specified point and determines if the difference between those two numbers exceeds a threshold.

In [ ]:
def rate_of_change_test(ts: pd.Series, threshold_value: float, previous_number_of_points: int = 5) -> pd.DataFrame:
    """
    The Rate of Change Test compares determines if two values exeed a threshood.
    
    Specifically, the test compares the mean value of the N-m points, where m
    indicates the number of previous points, to the Nth point and 
    determines if the difference between those two numbers exceeds a threshold.

    Parameters
    ----------
    ts : pd.Series
        A pandas series with a datetime index.

    threshold_value : float
        The threshold value for the difference.

    previous_number_of_points : int, optional
        The number of previous points to consider for the mean calculation. Default is 5.

    Returns
    -------
    pandas.DataFrame
        Outliers flagged as True or False.
    
    Examples
    --------
    >>> df_out = spike_detection_test(ts)

    """
    if ts.empty:
        raise ValueError('No input data.')
    mean_previous = ts.rolling(window=previous_number_of_points).mean()
    difference = ts - mean_previous
    output_ts = difference.abs() > threshold_value
    return output_ts

In [ ]:
output_ts = rate_of_change_test(ts=df.VALUE, threshold_value=1.1)

In [ ]:
df_rate_of_change = pd.DataFrame({"TIME":df.index, "VALUE":output_ts, "pH":df.VALUE})

In [ ]:
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(15,5))
sns.scatterplot(df_rate_of_change, x='TIME', y='pH', hue='VALUE', edgecolor=None).set_title('Outlier Detection Method: \nRate of Change')
ax.set(ylabel='pH')
#fig.savefig('rateofchange.png', dpi=300, transparent=True, bbox_inches='tight')